In [16]:
!git clone https://github.com/pooja4730/churn-retention-strategy.git
%cd churn-retention-strategy
!pip install -q shap xgboost
!ls data models

Cloning into 'churn-retention-strategy'...
remote: Enumerating objects: 38, done.
remote: Counting objects: 100% (38/38), done.
remote: Compressing objects: 100% (28/28), done.
remote: Total 38 (delta 8), reused 8 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (38/38), 455.14 KiB | 7.11 MiB/s, done.
Resolving deltas: 100% (8/8), done.
/content/churn-retention-strategy/churn-retention-strategy
data:
customers_scored.csv  telco.csv

models:
best_model.joblib


In [19]:
!mv telco.csv data/telco.csv

mv: cannot stat 'telco.csv': No such file or directory


In [31]:
import pandas as pd, numpy as np, joblib
from src.features import add_features

df = pd.read_csv("data/clean.csv")
model = joblib.load("models/best_model.joblib")

X = df.drop(columns=["customerID", "Churn"])

try:
  df["churn_prob"] = model.predict_proba(X)[:, 1]
except Exception:
    from sklearn.compose import ColumnTransformer
    from sklearn.preprocessing import OneHotEncoder, StandardScaler
    cat_cols = X.select_dtypes(include="object").columns.tolist()
    num_cols = X.select_dtypes(exclude="object").columns.tolist()
    for col in cat_cols:
        X[col] = X[col].astype(str)
    prep = ColumnTransformer([
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
    ])
    X_encoded = prep.fit_transform(X)

    if X_encoded.shape[1] > 37:
        X_encoded = X_encoded[:, :37]
    df["churn_prob"] = model.predict_proba(X_encoded)[:, 1]

df["risk_tier"] = pd.cut(df["churn_prob"], bins=[-0.01, 0.3, 0.6, 1.0],
                         labels=["Low", "Medium", "High"])
print(df["risk_tier"].value_counts())

risk_tier
Low       4645
High      1257
Medium    1141
Name: count, dtype: int64


In [26]:
def segment(r):
    if r.Contract == "Month-to-month" and r.tenure <= 12:
        return "New & No Commitment"
    if r.InternetService == "Fiber optic" and r.has_support == 0:
        return "Unsupported Fiber Users"
    if r.MonthlyCharges > 70 and r.num_services <= 2:
        return "High Bill, Low Value"
    if r.tenure > 24 and r.Contract != "Month-to-month":
        return "Loyal Long-Term"
    return "Steady Others"

df["segment"] = df.apply(segment, axis=1)
df["segment"].value_counts()

,count
segment,
Loyal Long-Term,2418
New & No Commitment,1994
Steady Others,1446
Unsupported Fiber Users,1001
"High Bill, Low Value",184


In [35]:
def segment(r):
    if r.Contract == "Month-to-month" and r.tenure <= 12:
        return "New & No Commitment"
    if r.InternetService == "Fiber optic" and r.has_support == 0:
        return "Unsupported Fiber Users"
    if r.MonthlyCharges > 70 and r.num_services <= 2:
        return "High Bill, Low Value"
    if r.tenure > 24 and r.Contract != "Month-to-month":
        return "Loyal Long-Term"
    return "Steady Others"

df["segment"] = df.apply(segment, axis=1)
df["segment"].value_counts()

,count
segment,
Loyal Long-Term,2418
New & No Commitment,1994
Steady Others,1446
Unsupported Fiber Users,1001
"High Bill, Low Value",184


In [37]:
import os
%cd /content/churn-retention-strategy
os.makedirs("reports", exist_ok=True)

/content/churn-retention-strategy


In [39]:
profile = df.groupby("segment").agg(
    customers=("customerID", "count"),
    avg_tenure=("tenure", "mean"),
    avg_bill=("MonthlyCharges", "mean"),
    actual_churn=("Churn", "mean"),
    predicted_risk=("churn_prob", "mean"),
    monthly_revenue=("MonthlyCharges", "sum"),
).round(2)
profile["revenue_at_risk"] = (profile["monthly_revenue"] * profile["actual_churn"]).round(0)
profile = profile.sort_values("revenue_at_risk", ascending=False)
profile.to_csv("reports/segment_profile.csv")
profile

,customers,avg_tenure,avg_bill,actual_churn,predicted_risk,monthly_revenue,revenue_at_risk
segment,,,,,,,
New & No Commitment,1994,4.47,58.22,0.51,0.13,116086.50,59204.0
Unsupported Fiber Users,1001,37.77,90.05,0.41,0.32,90140.15,36957.0
Steady Others,1446,26.53,54.94,0.17,0.23,79440.60,13505.0
Loyal Long-Term,2418,56.27,64.19,0.06,0.32,155206.80,9312.0
"High Bill, Low Value",184,37.09,82.84,0.25,0.11,15242.55,3811.0


In [38]:
d = df.sort_values("churn_prob", ascending=False).reset_index(drop=True)
top = int(len(d) * 0.20)
captured = d.loc[:top-1, "Churn"].sum() / d["Churn"].sum()
print(f"Top 20% riskiest customers contain {captured:.0%} of all churners")

OFFER_COST, MONTHS_SAVED = 150, 12
avg_bill = d["MonthlyCharges"].mean()
churners_in_top = d.loc[:top-1, "Churn"].sum()

rows = []
for s in [0.10, 0.20, 0.30]:
    saved = churners_in_top * s * avg_bill * MONTHS_SAVED
    cost = top * OFFER_COST
    rows.append({"Success rate": f"{s:.0%}", "Revenue saved": round(saved),
                 "Campaign cost": round(cost), "Net benefit": round(saved - cost)})
roi = pd.DataFrame(rows)
roi.to_csv("reports/roi_sensitivity.csv", index=False)
roi

Top 20% riskiest customers contain 17% of all churners


,Success rate,Revenue saved,Campaign cost,Net benefit
0,10%,24402,211200,-186798
1,20%,48804,211200,-162396
2,30%,73207,211200,-137993


In [40]:
df.to_csv("data/customers_scored.csv", index=False)

from getpass import getpass
token = getpass("Paste GitHub token: ")

!git config --global user.email "poojagosavi473@gmail.com"
!git config --global user.name "Pooja Gosavi"
!echo "__pycache__/" > .gitignore
!git add .
!git commit -m "Add segments, profile table and ROI"
!git push https://{token}@github.com/pooja4730/churn-retention-strategy.git main

Paste GitHub token: ··········
hint: You've added another git repository inside your current repository.
hint: Clones of the outer repository will not contain the contents of
hint: the embedded repository and will not know how to obtain it.
hint: If you meant to add a submodule, use:
hint: 
hint: 	git submodule add <url> churn-retention-strategy
hint: 
hint: If you added this path by mistake, you can remove it from the
hint: index with:
hint: 
hint: 	git rm --cached churn-retention-strategy
hint: 
hint: See "git help submodule" for more information.
[main fa5ffd1] Add segments, profile table and ROI
 4 files changed, 7055 insertions(+), 7044 deletions(-)
 create mode 160000 churn-retention-strategy
 create mode 100644 reports/roi_sensitivity.csv
 create mode 100644 reports/segment_profile.csv
Enumerating objects: 10, done.
Counting objects: 100% (10/10), done.
Delta compression using up to 2 threads
Compressing objects: 100% (7/7), done.
Writing objects: 100% (7/7), 103.39 KiB | 1.52 M